# Challenge 1 (Binary): descriptive analysis

**Nevus vs others** (training set: >15,000 images, about half nevus).

This notebook covers the training and validation images:

1. Image inventory per split and class
2. Class distribution bar charts (counts and %)
3. What the training class counts look like with different balancing strategies
4. Image properties (resolution, file size, mean colour)
5. Example images (5 per class)

Expected layout: `Binary/<train|val>/<class>/*.jpg`. Folder and class names are detected automatically.

In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

pd.set_option("display.max_columns", 50)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 0. Configuration

In [ ]:
# Project root = folder that contains the Binary/ and Multiclass/ data folders.
# If the notebook runs from notebooks/, Binary/ or Multiclass/, go one level up.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name in ("notebooks", "Binary", "Multiclass") else Path.cwd()

TASK = "Binary"   # this notebook only analyses one challenge
TASK_DIRS = {TASK: PROJECT_ROOT / TASK}

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

# Images sampled per (task, split, class) when computing image properties.
# Set to None to use every image (slower: 20k+ files).
PROPS_SAMPLE_PER_CLASS = 400

N_EXAMPLES = 5   # example images shown per class

for t, p in TASK_DIRS.items():
    print(f"{t:<11} -> {p}  (exists: {p.exists()})")

## 1. Image inventory

In [ ]:
def normalise_split(name: str) -> str:
    n = name.lower()
    if n.startswith("train"):
        return "train"
    if n.startswith("val"):
        return "val"
    if n.startswith("test"):
        return "test"
    return n

def build_index(task_dirs):
    rows = []
    for task, root in task_dirs.items():
        if not root.exists():
            print(f"[warning] {root} not found, skipping")
            continue
        for split_dir in sorted(p for p in root.iterdir() if p.is_dir()):
            split = normalise_split(split_dir.name)
            for f in split_dir.rglob("*"):
                if f.suffix.lower() in IMG_EXT and f.is_file():
                    rel = f.relative_to(split_dir).parts
                    cls = rel[0] if len(rel) > 1 else "(no class folder)"
                    rows.append({"task": task, "split": split, "class": cls, "path": f})
    df = pd.DataFrame(rows)
    if not df.empty:
        df["class"] = df["class"].str.lower()
    return df

df = build_index(TASK_DIRS)
print(f"Total images found: {len(df):,}")
df.head()

In [ ]:
counts = (df.groupby(["task", "split", "class"]).size()
            .rename("n").reset_index())
counts["%"] = (counts["n"] / counts.groupby(["task", "split"])["n"].transform("sum") * 100).round(1)

summary = counts.pivot_table(index=["task", "class"], columns="split", values=["n", "%"], fill_value=0)
summary

In [ ]:
totals = df.groupby(["task", "split"]).size().unstack(fill_value=0)
totals["total"] = totals.sum(axis=1)
totals

## 2. Class distribution

Absolute counts and class proportions for each split. If train and val have similar proportions, the validation set is representative of the training distribution.

In [ ]:
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860"]

def class_colors(classes):
    return {c: PALETTE[i % len(PALETTE)] for i, c in enumerate(sorted(classes))}

def plot_distribution(task):
    sub = counts[counts.task == task]
    if sub.empty:
        print(f"No data for {task}")
        return
    splits = [s for s in ["train", "val", "test"] if s in sub.split.unique()]
    classes = sorted(sub["class"].unique())
    colors = class_colors(classes)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

    # (a) absolute counts, grouped by split
    w = 0.8 / len(classes)
    x = np.arange(len(splits))
    for i, c in enumerate(classes):
        vals = [sub[(sub.split == s) & (sub["class"] == c)]["n"].sum() for s in splits]
        bars = axes[0].bar(x + i * w - 0.4 + w / 2, vals, w, label=c, color=colors[c])
        axes[0].bar_label(bars, fmt="%d", fontsize=8, padding=2)
    axes[0].set_xticks(x, splits)
    axes[0].set_ylabel("# images")
    axes[0].set_title(f"{task}: images per class")
    axes[0].legend(frameon=False)

    # (b) proportions (stacked 100 %)
    bottom = np.zeros(len(splits))
    for c in classes:
        pct = np.array([sub[(sub.split == s) & (sub["class"] == c)]["%"].sum() for s in splits])
        axes[1].bar(splits, pct, bottom=bottom, color=colors[c], label=c)
        for xi, (b, p) in enumerate(zip(bottom, pct)):
            if p > 4:
                axes[1].text(xi, b + p / 2, f"{p:.1f}%", ha="center", va="center", color="white", fontsize=9)
        bottom += pct
    axes[1].set_ylim(0, 100)
    axes[1].set_ylabel("% of split")
    axes[1].set_title(f"{task}: class proportions")
    plt.tight_layout()
    plt.show()

    tr = sub[sub.split == "train"].set_index("class")["n"]
    if len(tr):
        print(f"Imbalance ratio in train (majority / minority): {tr.max() / tr.min():.2f}")

plot_distribution(TASK)

## 3. Class balancing strategies (training set)

The training set can be rebalanced in several ways before or during training. The charts below show what the class counts become with each strategy.

- **Original**: no balancing
- **Random undersampling**: every class is reduced to the size of the smallest class. This loses data.
- **Random oversampling**: every class is increased to the size of the largest class by repeating images (or adding augmented copies). This can cause overfitting.
- **Class weights**: the data stays the same, but each class gets a weight in the loss/classifier, `w_c = N / (K * n_c)` (sklearn `class_weight="balanced"`). The *effective* count is `n_c * w_c`.

In [ ]:
def balancing_table(task):
    tr = counts[(counts.task == task) & (counts.split == "train")].set_index("class")["n"].sort_index()
    if tr.empty:
        return None, None
    N, K = tr.sum(), len(tr)
    weights = N / (K * tr)
    table = pd.DataFrame({
        "original": tr,
        "undersampling": tr.min(),
        "oversampling": tr.max(),
        "class-weighted (effective)": (tr * weights).round().astype(int),
    })
    return table, weights.rename("class_weight").round(3)

def plot_balancing(task):
    table, weights = balancing_table(task)
    if table is None:
        print(f"No training data for {task}")
        return
    colors = class_colors(table.index)
    strategies = table.columns
    fig, axes = plt.subplots(1, len(strategies), figsize=(4 * len(strategies), 3.8), sharey=True)
    for ax, s in zip(axes, strategies):
        bars = ax.bar(table.index, table[s], color=[colors[c] for c in table.index])
        ax.bar_label(bars, fmt="%d", fontsize=8, padding=2)
        ax.set_title(s, fontsize=10)
        ax.tick_params(axis="x", rotation=30)
    axes[0].set_ylabel("# training samples")
    fig.suptitle(f"{task}: training set under different balancing strategies", y=1.02)
    plt.tight_layout()
    plt.show()
    display(table)
    display(weights.to_frame())

plot_balancing(TASK)

## 4. Image properties

Resolution, aspect ratio and file size are read from the image headers, which is fast. Mean RGB is computed on a downscaled copy of each image.
To keep this quick, a random sample of `PROPS_SAMPLE_PER_CLASS` images per task/split/class is used.

Different resolutions and colour profiles point to the different acquisition sources (HAM10000, BCN_20000, MSK). This affects preprocessing choices, such as the resize strategy and colour constancy.

In [ ]:
if PROPS_SAMPLE_PER_CLASS is None:
    sampled = df
else:  # shuffle, then keep the first N of each (task, split, class) group
    sampled = (df.sample(frac=1, random_state=SEED)
                 .groupby(["task", "split", "class"]).head(PROPS_SAMPLE_PER_CLASS))

def image_props(path):
    with Image.open(path) as im:
        w, h = im.size
        small = im.convert("RGB").resize((64, 64))
    m = np.asarray(small, dtype=np.float32).reshape(-1, 3).mean(0)
    return w, h, path.stat().st_size / 1024, *m

try:
    from tqdm.auto import tqdm
    it = tqdm(sampled["path"], desc="Reading images")
except ImportError:
    it = sampled["path"]

props = pd.DataFrame([image_props(p) for p in it],
                     columns=["width", "height", "size_kb", "mean_R", "mean_G", "mean_B"],
                     index=sampled.index)
props = pd.concat([sampled[["task", "split", "class"]], props], axis=1)
props["aspect"] = props["width"] / props["height"]
props["resolution"] = props["width"].astype(str) + "x" + props["height"].astype(str)

props.groupby(["task", "class"])[["width", "height", "aspect", "size_kb", "mean_R", "mean_G", "mean_B"]].agg(["mean", "std"]).round(1)

In [ ]:
# Most frequent resolutions per task (different values = different acquisition sources)
(props.groupby(["task", "resolution"]).size().rename("n")
      .reset_index().sort_values(["task", "n"], ascending=[True, False])
      .groupby("task").head(8))

In [ ]:
def plot_props(task):
    sub = props[props.task == task]
    if sub.empty:
        return
    classes = sorted(sub["class"].unique())
    colors = class_colors(classes)
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
    for c in classes:
        s = sub[sub["class"] == c]
        axes[0].scatter(s["width"], s["height"], s=8, alpha=0.4, color=colors[c], label=c)
        axes[1].hist(s["size_kb"], bins=40, alpha=0.5, color=colors[c], label=c, density=True)
    axes[0].set(xlabel="width (px)", ylabel="height (px)", title="Resolution")
    axes[1].set(xlabel="file size (KB)", ylabel="density", title="File size")
    axes[0].legend(frameon=False, markerscale=2)

    means = sub.groupby("class")[["mean_R", "mean_G", "mean_B"]].mean().loc[classes]
    x = np.arange(len(classes))
    for i, (ch, col) in enumerate(zip(["mean_R", "mean_G", "mean_B"], ["#d62728", "#2ca02c", "#1f77b4"])):
        axes[2].bar(x + (i - 1) * 0.25, means[ch], 0.25, color=col, label=ch[-1])
    axes[2].set_xticks(x, classes, rotation=30)
    axes[2].set(ylabel="mean intensity (0-255)", title="Mean colour per class")
    axes[2].legend(frameon=False)
    fig.suptitle(task, y=1.02)
    plt.tight_layout()
    plt.show()

plot_props(TASK)

## 5. Example images

`N_EXAMPLES` random training images per class. Change `SEED` to see different ones.

In [ ]:
def show_examples(task, split="train", n=N_EXAMPLES, seed=SEED):
    sub = df[(df.task == task) & (df.split == split)]
    if sub.empty:
        print(f"No images for {task}/{split}")
        return
    classes = sorted(sub["class"].unique())
    fig, axes = plt.subplots(len(classes), n, figsize=(2.6 * n, 2.6 * len(classes)))
    axes = np.atleast_2d(axes)
    for r, c in enumerate(classes):
        paths = sub[sub["class"] == c]["path"].sample(min(n, (sub["class"] == c).sum()), random_state=seed).tolist()
        for k in range(n):
            ax = axes[r, k]
            ax.axis("off")
            if k < len(paths):
                with Image.open(paths[k]) as im:
                    ax.imshow(im.convert("RGB"))
                    ax.set_title(f"{im.size[0]}x{im.size[1]}", fontsize=8)
        axes[r, 0].text(-0.08, 0.5, c, transform=axes[r, 0].transAxes, rotation=90,
                        ha="right", va="center", fontsize=12, fontweight="bold")
    fig.suptitle(f"{task}: {split} examples", fontsize=13)
    plt.tight_layout()
    plt.show()

show_examples(TASK)

## 6. Observations

*(Fill in after running.)*

- Is the nevus / others split really close to 50/50? Do train and val have the same proportions?
- The *others* class mixes several lesion types (melanoma, BCC, SCC, keratoses...), so it is visually heterogeneous
- Acquisition: variety of resolutions/sources, which suggests resizing and colour normalisation (e.g. Shades of Gray)
- Artefacts in the examples: hair, dark corners/vignetting, rulers, gel bubbles. These call for preprocessing such as DullRazor and vignette cropping.